# Notebook 3 — Model Training
**Credit Card Fraud Detection | IS525E Data Science for Business**

Models trained and compared:
1. Logistic Regression (baseline)
2. Random Forest
3. XGBoost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

%matplotlib inline

## 1. Load Processed Data

In [ ]:
train = pd.read_csv('../data/processed/train_resampled.csv')
test  = pd.read_csv('../data/processed/test.csv')

X_train = train.drop(columns=['Class'])
y_train = train['Class']
X_test  = test.drop(columns=['Class'])
y_test  = test['Class']

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 2. Train Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)
}

os.makedirs('../reports/models', exist_ok=True)
results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    results[name] = {'model': model, 'y_pred': y_pred, 'y_proba': y_proba, 'auc': auc}
    print(f'  AUC-ROC: {auc:.4f}')
    # Save model
    with open(f'../reports/models/{name.replace(" ", "_").lower()}.pkl', 'wb') as f:
        pickle.dump(model, f)

print('\nAll models trained and saved.')

## 3. Classification Reports

In [ ]:
for name, res in results.items():
    print(f'\n=== {name} ===')
    print(classification_report(y_test, res['y_pred'], target_names=['Legitimate', 'Fraud']))

## 4. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['auc']:.4f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Fraud Detection Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/roc_curves.png', dpi=150)
plt.show()

**Next step:** Notebook 04 — Evaluation & Business Interpretation